# Analytical Placement v4 — smart expansion + soft macro spreading

Fixes the two problems observed in `train_3.ipynb`:

1. **Edge piling** — `scale=1.5` from canvas center pushed every
   edge-adjacent macro over the boundary and the clamp created a
   pancake against the wall. Fix: compute the *maximum safe scale*
   that doesn't require any clamping, applied around the centroid of
   *movable* hard macros (not the canvas center).
2. **Soft macros (standard cell clusters) collapsing to center** — the
   overlap penalty and springs only act on hard macros, so soft macros
   feel *only* the wirelength gradient, which pulls them all toward
   their connection centroids. Fix: add a density spreading penalty
   applied to the soft-macro slice.

Loss = smooth WL  +  hard-overlap  +  spring  +  **soft-density**  +  canvas

## 1. Setup

In [ ]:
import os, sys, math
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR
while not (REPO_ROOT / "macro_place").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

ANALYTICAL_DIR = REPO_ROOT / "submissions" / "analytical"
CHECKPOINT_DIR = ANALYTICAL_DIR / "checkpoints_v4"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

import torch
import torch.nn.functional as F

def select_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = select_device()
if DEVICE.type == "cpu":
    torch.set_num_threads(os.cpu_count() or 1)

print(f"Repo root: {REPO_ROOT}")
print(f"Device:    {DEVICE}")

## 2. Imports

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display as ipy_display

from macro_place.loader import load_benchmark_from_dir
from submissions.analytical import losses as L
from submissions.core.eval import evaluate as tilos_evaluate

## 3. Load benchmark

In [ ]:
BENCHMARK_NAME = "ibm01"
BENCHMARK_DIR = REPO_ROOT / "external" / "MacroPlacement" / "Testcases" / "ICCAD04" / BENCHMARK_NAME

benchmark, plc = load_benchmark_from_dir(str(BENCHMARK_DIR))
print(benchmark)
print(f"  hard macros: {benchmark.num_hard_macros}")
print(f"  soft macros: {benchmark.num_soft_macros}")

## 4. Smart expansion (no clamping)

Compute the centroid of movable hard macros, then find the maximum
uniform scale factor that keeps *every* hard macro inside the canvas
without clamping. Apply that scale (capped at `target_scale`).

Why this is better than the v3 approach:

- Centroid-based expansion preserves relative topology better when the
  baseline is asymmetric (e.g., macros biased to one side).
- Adaptive cap means edge macros never get pushed against the wall.
- The function returns the actual scale used, so you know if your
  `target_scale` was achievable or had to be clipped.

In [ ]:
def smart_expand(benchmark, target_scale=1.5):
    """Expand around movable centroid using the max scale that doesn't clamp."""
    num_hard = benchmark.num_hard_macros
    positions = benchmark.macro_positions.clone().float()
    sizes     = benchmark.macro_sizes.float()
    fixed     = benchmark.macro_fixed[:num_hard]
    movable   = ~fixed

    if not movable.any():
        return positions, 1.0

    # Centroid of movable hard macros (not the canvas center).
    pos_m  = positions[:num_hard][movable]              # [M, 2]
    half_m = sizes[:num_hard][movable] / 2.0            # [M, 2]
    centroid = pos_m.mean(dim=0)

    canvas = torch.tensor(
        [benchmark.canvas_width, benchmark.canvas_height], dtype=positions.dtype
    )

    # offset = pos - centroid;  constraint: half <= centroid + s*offset <= canvas - half
    offset = pos_m - centroid
    # Per-axis max scale that keeps each macro inside the canvas.
    eps = 1e-9
    pos_offset = offset.clamp(min=eps)                  # safe divide for positive
    neg_offset = offset.clamp(max=-eps)                 # safe divide for negative
    upper = (canvas - half_m - centroid) / pos_offset   # binding when offset > 0
    lower = (half_m - centroid)          / neg_offset   # binding when offset < 0
    # Pick the active constraint per element by sign of offset.
    constraint = torch.where(offset > 0, upper, lower)
    # Where offset is ~0, no constraint binds.
    constraint = torch.where(offset.abs() < 1e-6,
                             torch.full_like(constraint, float("inf")),
                             constraint)
    max_safe = float(constraint.min().clamp(min=1.0))   # never shrink
    actual_scale = min(float(target_scale), max_safe)

    # Apply uniform scale around centroid to movable macros only.
    expanded = centroid + actual_scale * offset
    new_positions = positions.clone()
    new_hard = positions[:num_hard].clone()
    new_hard[movable] = expanded
    new_positions[:num_hard] = new_hard

    return new_positions, actual_scale


init_positions, used_scale = smart_expand(benchmark, target_scale=1.5)
print(f"Requested target_scale=1.50, actually used scale={used_scale:.3f}")
if used_scale < 1.5:
    print("  (capped because some macro would have exited the canvas)")

## 5. Build topology springs (same as v3)

In [ ]:
def build_springs(benchmark, k=8):
    num_hard = benchmark.num_hard_macros
    pos = benchmark.macro_positions[:num_hard].float()
    diff = pos.unsqueeze(0) - pos.unsqueeze(1)
    dist = (diff * diff).sum(-1).sqrt() + torch.eye(num_hard) * 1.0e9
    topk_idx = dist.topk(k, dim=-1, largest=False).indices

    edges = set()
    for i in range(num_hard):
        for jj in range(k):
            j = int(topk_idx[i, jj])
            edges.add((min(i, j), max(i, j)))
    edges = sorted(edges)

    si = torch.tensor([e[0] for e in edges], dtype=torch.long)
    sj = torch.tensor([e[1] for e in edges], dtype=torch.long)
    rl = (pos[si] - pos[sj]).norm(dim=-1)
    return si, sj, rl


def spring_loss(positions, spring_i, spring_j, rest_length, stiffness_param):
    delta = positions[spring_i] - positions[spring_j]
    current_dist = (delta * delta).sum(-1).clamp(min=1e-12).sqrt()
    extension = current_dist - rest_length
    eff = F.softplus(stiffness_param)
    return (eff * extension * extension).sum()


K_NEIGHBORS = 8
spring_i, spring_j, rest_length = build_springs(benchmark, k=K_NEIGHBORS)
print(f"Built {len(spring_i)} springs (top-{K_NEIGHBORS} neighbors).")

## 6. Soft-macro spreading penalty

Reuses `L.density_penalty` but applied to the **soft macro slice**
only. Each soft macro contributes its area to a bilinear-interpolated
bin grid; bins exceeding the soft-macro average get penalized.

This is the missing force that prevents standard-cell clusters from
collapsing to their wirelength centroid.

In [ ]:
def soft_density_penalty(positions, sizes, num_hard, num_macros,
                         canvas_width, canvas_height, n_bins=12):
    """Bin-based spreading penalty for soft macros only."""
    n_soft = num_macros - num_hard
    if n_soft < 2:
        return torch.tensor(0.0, device=positions.device, dtype=positions.dtype)
    soft_pos = positions[num_hard:num_macros]
    soft_sz  = sizes[num_hard:num_macros]
    # L.density_penalty indexes positions[:num_hard], so passing the soft
    # slice with num_hard=n_soft applies it to soft macros only.
    return L.density_penalty(
        soft_pos, soft_sz, n_soft, canvas_width, canvas_height, n_bins=n_bins
    )

## 7. Visualization (hard + soft shown separately)

In [ ]:
def plot_placement(ax, positions, benchmark, title="", show_soft=True,
                   show_springs=False, spring_i=None, spring_j=None,
                   max_springs=400):
    ax.clear()
    ax.set_xlim(0, benchmark.canvas_width)
    ax.set_ylim(0, benchmark.canvas_height)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.set_xlabel("x (μm)")
    ax.set_ylabel("y (μm)")

    pos = positions.detach().cpu().numpy()
    sizes = benchmark.macro_sizes.cpu().numpy()
    fixed = benchmark.macro_fixed.cpu().numpy()
    num_hard = benchmark.num_hard_macros
    num_macros = benchmark.num_macros

    # Springs first (under macros).
    if show_springs and spring_i is not None:
        n = len(spring_i)
        idx = np.arange(n) if n <= max_springs else np.random.choice(n, max_springs, replace=False)
        si = spring_i.cpu().numpy(); sj = spring_j.cpu().numpy()
        for k in idx:
            i, j = int(si[k]), int(sj[k])
            ax.plot([pos[i, 0], pos[j, 0]], [pos[i, 1], pos[j, 1]],
                    color="crimson", alpha=0.12, linewidth=0.3, zorder=1)

    # Soft macros as small dots (translucent green).
    if show_soft and num_macros > num_hard:
        soft_pos = pos[num_hard:num_macros]
        ax.scatter(
            soft_pos[:, 0], soft_pos[:, 1],
            s=4, c="seagreen", alpha=0.25, zorder=2, label="soft",
        )

    # Hard macros as filled rectangles.
    for i in range(num_hard):
        x = pos[i, 0] - sizes[i, 0] / 2.0
        y = pos[i, 1] - sizes[i, 1] / 2.0
        color = "lightgray" if fixed[i] else "steelblue"
        ax.add_patch(mpatches.Rectangle(
            (x, y), sizes[i, 0], sizes[i, 1],
            facecolor=color, edgecolor="black",
            alpha=0.65, linewidth=0.4, zorder=3,
        ))


def plot_loss(ax, history):
    ax.clear()
    if not history["step"]:
        return
    for key, label in [("wl", "smooth WL"), ("overlap", "hard overlap"),
                       ("spring", "spring"), ("soft_density", "soft density"),
                       ("total", "total")]:
        if key in history and any(v > 0 for v in history[key]):
            lw = 2 if key == "total" else 1
            ax.plot(history["step"], history[key], label=label, linewidth=lw)
    ax.set_yscale("log")
    ax.set_xlabel("step")
    ax.set_ylabel("loss (log)")
    ax.set_title("Loss components")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right")


# Compare baseline vs smart-expanded init.
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_placement(axes[0], benchmark.macro_positions, benchmark,
               title=f"{benchmark.name} — baseline", show_soft=True)
plot_placement(axes[1], init_positions, benchmark,
               title=f"{benchmark.name} — smart-expanded ({used_scale:.2f}×) + springs",
               show_soft=True, show_springs=True,
               spring_i=spring_i, spring_j=spring_j)
plt.tight_layout(); plt.show()

## 8. Training function (with soft-density + resume support)

In [ ]:
def train_v4(
    benchmark,
    plc,
    init_positions,
    spring_i, spring_j, rest_length,
    *,
    n_epochs=100,
    steps_per_epoch=100,
    lr=3e-3,
    lr_stiffness=1e-2,
    gamma_start=0.01,
    gamma_end=0.002,
    w_overlap=80.0,            # hard macros only
    w_canvas=200.0,
    w_spring_start=200.0,
    w_spring_end=10.0,
    w_soft_density=500.0,      # NEW: prevent soft-macro collapse
    n_bins_soft=12,            # finer grid for ~900 soft macros
    checkpoint_dir=CHECKPOINT_DIR,
    device=DEVICE,
    seed=42,
    resume_from=None,
):
    torch.manual_seed(seed)
    total_steps = n_epochs * steps_per_epoch

    if resume_from is not None:
        ckpt = torch.load(resume_from, map_location="cpu", weights_only=False)
        positions = ckpt["positions"].to(device).clone().requires_grad_(True)
        stiffness = ckpt["stiffness"].to(device).clone().requires_grad_(True)
        history   = ckpt["history"]
        start_step = ckpt["step"]
        print(f"Resuming from {resume_from}  (step {start_step})")
    else:
        positions = init_positions.clone().to(device).requires_grad_(True)
        stiffness = torch.full(
            (len(spring_i),), math.log(math.e - 1.0),
            device=device, dtype=torch.float32,
        ).requires_grad_(True)
        history = {"step": [], "wl": [], "overlap": [], "spring": [],
                   "soft_density": [], "canvas": [], "total": [], "w_spring": []}
        start_step = 0

    sizes_dev = benchmark.macro_sizes.to(device, dtype=torch.float32)
    fixed_dev = benchmark.macro_fixed.to(device)
    anchor    = benchmark.macro_positions.to(device, dtype=torch.float32)
    spring_i_d, spring_j_d = spring_i.to(device), spring_j.to(device)
    rest_len_d = rest_length.to(device)

    padded_nets, mask, net_weights = L.prepare_net_tensors(benchmark, device)
    n_pos = positions.shape[0]
    out_of_bounds = padded_nets >= n_pos
    padded_nets = padded_nets.clamp(max=n_pos - 1)
    mask = mask & ~out_of_bounds

    optimizer = torch.optim.Adam([
        {"params": [positions], "lr": lr},
        {"params": [stiffness], "lr": lr_stiffness},
    ])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_steps, eta_min=lr / 100
    )
    if resume_from is not None and "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
        if "scheduler_state" in ckpt:
            scheduler.load_state_dict(ckpt["scheduler_state"])

    best_total = float("inf")
    best_positions = positions.detach().cpu().clone()
    epoch_log = []

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig_handle = ipy_display(fig, display_id=True)
    t0 = time.perf_counter()

    def lerp(s, e, frac):
        return s + (e - s) * max(0.0, min(1.0, frac))

    for step in range(start_step + 1, start_step + total_steps + 1):
        frac = (step - start_step) / total_steps
        gamma    = lerp(gamma_start,    gamma_end,    frac)
        w_spring = lerp(w_spring_start, w_spring_end, frac)

        optimizer.zero_grad()

        wl_term      = L.smooth_hpwl(positions, padded_nets, mask, net_weights, gamma=gamma)
        overlap_term = L.overlap_penalty(positions, sizes_dev, benchmark.num_hard_macros)
        canvas_term  = L.canvas_penalty(
            positions, sizes_dev, benchmark.canvas_width, benchmark.canvas_height
        )
        spring_term  = spring_loss(positions, spring_i_d, spring_j_d, rest_len_d, stiffness)
        soft_density_term = soft_density_penalty(
            positions, sizes_dev,
            benchmark.num_hard_macros, benchmark.num_macros,
            benchmark.canvas_width, benchmark.canvas_height,
            n_bins=n_bins_soft,
        )
        stiffness_reg = 1e-4 * (stiffness ** 2).sum()

        loss = (
            wl_term
            + w_overlap       * overlap_term
            + w_canvas        * canvas_term
            + w_spring        * spring_term
            + w_soft_density  * soft_density_term
            + stiffness_reg
        )
        loss.backward()

        with torch.no_grad():
            positions.grad[fixed_dev] = 0.0
        optimizer.step()
        scheduler.step()
        with torch.no_grad():
            positions.data[fixed_dev] = anchor[fixed_dev]

        loss_f = float(loss.detach())
        if loss_f < best_total:
            best_total = loss_f
            best_positions = positions.detach().cpu().clone()

        history["step"].append(step)
        history["wl"].append(float(wl_term.detach()))
        history["overlap"].append(max(float(overlap_term.detach()), 1e-12))
        history["spring"].append(max(float(spring_term.detach()), 1e-12))
        history["soft_density"].append(max(float(soft_density_term.detach()), 1e-12))
        history["canvas"].append(max(float(canvas_term.detach()), 1e-12))
        history["total"].append(loss_f)
        history["w_spring"].append(w_spring)

        if step % steps_per_epoch == 0:
            epoch_idx = step // steps_per_epoch
            elapsed = time.perf_counter() - t0

            plot_placement(
                axes[0], positions, benchmark,
                title=(
                    f"{benchmark.name}  epoch {epoch_idx}  "
                    f"γ={gamma:.4f}  w_spring={w_spring:.1f}"
                ),
                show_soft=True,
                show_springs=True,
                spring_i=spring_i, spring_j=spring_j,
            )
            plot_loss(axes[1], history)
            fig.tight_layout()
            fig_handle.update(fig)

            stiff = F.softplus(stiffness).detach()
            print(
                f"epoch {epoch_idx:>3d}  "
                f"total={loss_f:.2f}  wl={float(wl_term.detach()):.2f}  "
                f"spring={float(spring_term.detach()):.2f}  "
                f"soft_d={float(soft_density_term.detach()):.4f}  "
                f"hard_olap={float(overlap_term.detach()):.4f}  "
                f"stiff[μ]={float(stiff.mean()):.2f}±{float(stiff.std()):.2f}  "
                f"elapsed={elapsed:.1f}s"
            )

            ckpt_path = checkpoint_dir / f"{benchmark.name}_v4_epoch{epoch_idx:03d}.pt"
            torch.save({
                "benchmark": benchmark.name,
                "epoch": epoch_idx, "step":  step,
                "positions":      positions.detach().cpu(),
                "best_positions": best_positions,
                "stiffness":      stiffness.detach().cpu(),
                "spring_i":       spring_i,
                "spring_j":       spring_j,
                "rest_length":    rest_length,
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "history": history,
                "config": {
                    "lr": lr, "lr_stiffness": lr_stiffness,
                    "w_overlap": w_overlap, "w_canvas": w_canvas,
                    "w_spring_start": w_spring_start, "w_spring_end": w_spring_end,
                    "w_soft_density": w_soft_density, "n_bins_soft": n_bins_soft,
                    "used_scale": used_scale, "seed": seed,
                },
            }, ckpt_path)
            epoch_log.append({
                "epoch": epoch_idx, "step": step,
                "path":  str(ckpt_path),
                "total": loss_f,
                "wl":    float(wl_term.detach()),
                "spring":       float(spring_term.detach()),
                "soft_density": float(soft_density_term.detach()),
            })

    plt.close(fig)
    return best_positions, stiffness.detach().cpu(), history, epoch_log

## 9. Run training — 100 epochs

In [ ]:
positions, stiffness, history, epoch_log = train_v4(
    benchmark, plc, init_positions,
    spring_i, spring_j, rest_length,
    n_epochs=100,
    steps_per_epoch=100,
    w_spring_start=200.0,
    w_spring_end=10.0,
    w_soft_density=500.0,
)

print("\nLast 5 epochs:")
for row in epoch_log[-5:]:
    print(f"  epoch {row['epoch']:>3d}  total={row['total']:.2f}  "
          f"wl={row['wl']:.2f}  spring={row['spring']:.2f}  "
          f"soft_d={row['soft_density']:.4f}")

## 10. Resume from latest checkpoint (optional)

In [ ]:
all_ckpts = sorted(CHECKPOINT_DIR.glob(f"{BENCHMARK_NAME}_v4_epoch*.pt"))
RESUME_PATH = all_ckpts[-1] if all_ckpts else None
print(f"Resuming from: {RESUME_PATH}")

if RESUME_PATH is not None:
    positions, stiffness, history, epoch_log = train_v4(
        benchmark, plc, init_positions,
        spring_i, spring_j, rest_length,
        n_epochs=50,                    # additional epochs
        steps_per_epoch=100,
        w_spring_start=10.0,            # already weak — keep weak
        w_spring_end=2.0,
        w_soft_density=500.0,
        resume_from=str(RESUME_PATH),
    )
    print("\nLast 5 epochs:")
    for row in epoch_log[-5:]:
        print(f"  epoch {row['epoch']:>3d}  total={row['total']:.2f}")

## 11. Legalize and evaluate with TILOS

In [ ]:
from submissions.analytical.placer import AnalyticalPlacer
from submissions.core.eval import visualize

placer = AnalyticalPlacer(n_steps=0, legalize_iters=600, device=DEVICE)
pos = positions.clone().to(DEVICE).requires_grad_(True)
sizes_dev = benchmark.macro_sizes.to(DEVICE, dtype=torch.float32)
fixed_dev = benchmark.macro_fixed.to(DEVICE)
pos = placer._clamp_to_canvas(pos, sizes_dev, benchmark)
pos = placer._restore_fixed(pos, benchmark, fixed_dev)
pos = placer._legalize_push_apart(pos, sizes_dev, benchmark, fixed_dev)
pos = placer._restore_fixed(pos, benchmark, fixed_dev).detach().cpu()

result = tilos_evaluate(pos, benchmark, plc)
print(
    f"Proxy: {result['proxy_cost']:.4f}  "
    f"WL: {result['wirelength_cost']:.4f}  "
    f"Density: {result['density_cost']:.4f}  "
    f"Congestion: {result['congestion_cost']:.4f}  "
    f"Overlaps: {result['overlap_count']}"
)

final_path = CHECKPOINT_DIR / f"{benchmark.name}_v4_final.pt"
torch.save({"positions": pos, "tilos": result, "stiffness": stiffness}, final_path)
visualize(pos, benchmark, plc=plc, save_path=str(CHECKPOINT_DIR / f"{benchmark.name}_v4_final.png"))
print(f"Saved -> {final_path}")

## 12. Tuning notes

**Smart expansion still capped too low?**
If `used_scale` came out below 1.2×, your baseline has macros pinned to
the canvas edges. Lower `target_scale` to 1.2 to avoid wasted compute,
or accept the small expansion.

**Soft macros still cluster?**
- Raise `w_soft_density` to 1000–2000
- Or raise `n_bins_soft` to 16 for finer-grained spreading

**Hard macros now spread too much (springs too weak)?**
- Raise `w_spring_end` from 10 → 30
- Or use a softer decay: `w_spring_start=150, w_spring_end=30`

**Final TILOS proxy still high after good smooth loss?**
- The smooth loss is a *surrogate* — large smooth-WL improvements can
  still produce a worse TILOS result if soft macros congregate or
  density imbalance increases. Check the per-term breakdown
  (`result['density_cost']`, `result['wirelength_cost']`) and bias the
  weights accordingly.